In [ ]:
import os
import numpy as np


test_feat_path = 'dataset/Full_dataset_processed_split/test_features.npy'
test_label_path = 'dataset/Full_dataset_processed_split/test_labels.npy'

examplar_dir = 'exemplar_data/final_nolin_act_processed' 

model_categories = ['Jet', 'Quark', 'Anomaly', 'BiPC', 'Cookie', 'Automlp', 'Particle']

ex_numpy_list = []

for category in model_categories:
    features_path = os.path.join(examplar_dir, f"{category}_processed/{category}_features.npy")
    labels_path = os.path.join(examplar_dir, f"{category}_processed/{category}_labels.npy")

    feat_temp = np.load(features_path)
    label_temp = np.load(labels_path)

    ex_numpy_list.append(
        (feat_temp, label_temp, category)
    )
    print(f" shape of {category} features: {feat_temp.shape}, labels: {label_temp.shape}")


print("shape of test features: ", np.load(test_feat_path).shape)
print("shape of test labels: ", np.load(test_label_path).shape)




 shape of Jet features: (124, 8, 18), labels: (124, 6)
 shape of Quark features: (126, 4, 18), labels: (126, 6)
 shape of Anomaly features: (133, 10, 18), labels: (133, 6)
 shape of BiPC features: (119, 12, 18), labels: (119, 6)
 shape of Cookie features: (130, 8, 18), labels: (130, 6)
 shape of Automlp features: (127, 8, 18), labels: (127, 6)
 shape of Particle features: (128, 8, 18), labels: (128, 6)
shape of test features:  (94430, 47, 18)
shape of test labels:  (94430, 6)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

analysis_plot_dir = 'Plotting_distribution_Oct1'

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Feature names based on the FEATURES dictionary from your code
FEATURE_NAMES = [
    "d_in1", "d_in2", "d_in3", "d_out1", "d_out2", "d_out3",
    "prec", "rf", "strategy", "layer_type", "activation_type",
    "filters", "kernel_size", "stride", "padding", "pooling",
    "batchnorm", "io_type"
]

LABEL_NAMES = ["cycles_max", "ff", "lut", "bram", "dsp", "interval_max"]

def load_data():
    """Load test and exemplar data"""
    # Load test data
    test_feat_path = 'dataset/Full_dataset_processed_split/test_features.npy'
    test_label_path = 'dataset/Full_dataset_processed_split/test_labels.npy'
    
    test_features = np.load(test_feat_path)
    test_labels = np.load(test_label_path)
    
    # Load exemplar data
    exemplar_dir = 'exemplar_data/final_nolin_act_processed'
    model_categories = ['Jet', 'Quark', 'Anomaly', 'BiPC', 'Cookie', 'Automlp', 'Particle']
    
    exemplar_data = {}
    for category in model_categories:
        features_path = os.path.join(exemplar_dir, f"{category}_processed/{category}_features.npy")
        labels_path = os.path.join(exemplar_dir, f"{category}_processed/{category}_labels.npy")
        
        features = np.load(features_path)
        labels = np.load(labels_path)
        
        exemplar_data[category] = {
            'features': features,
            'labels': labels
        }
        print(f"Loaded {category}: features shape {features.shape}, labels shape {labels.shape}")
    
    print(f"\nTest data: features shape {test_features.shape}, labels shape {test_labels.shape}")
    
    return test_features, test_labels, exemplar_data

def remove_padding(features):
    """Remove -1 padding from features and flatten to 2D array"""
    # Reshape to (n_samples * n_layers, n_features)
    n_models, n_layers, n_features = features.shape
    features_flat = features.reshape(-1, n_features)
    
    # Remove rows with -1 (padding)
    mask = ~np.any(features_flat == -1, axis=1)
    features_clean = features_flat[mask]
    
    return features_clean

def compute_feature_statistics(features_clean, name="Dataset"):
    """Compute statistics for each feature"""
    stats_dict = {
        'Dataset': name,
        'N_samples': len(features_clean)
    }
    
    for i, feat_name in enumerate(FEATURE_NAMES):
        feat_data = features_clean[:, i]
        stats_dict[f'{feat_name}_mean'] = np.mean(feat_data)
        stats_dict[f'{feat_name}_std'] = np.std(feat_data)
        stats_dict[f'{feat_name}_median'] = np.median(feat_data)
        stats_dict[f'{feat_name}_min'] = np.min(feat_data)
        stats_dict[f'{feat_name}_max'] = np.max(feat_data)
    
    return stats_dict

def plot_feature_distributions(test_features, exemplar_data, output_dir=analysis_plot_dir):
    """Plot distributions of all features comparing test vs exemplar datasets"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Remove padding from test features
    test_clean = remove_padding(test_features)
    
    # Create figure with subplots for all features
    n_features = len(FEATURE_NAMES)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
    axes = axes.flatten()
    
    for feat_idx, feat_name in enumerate(FEATURE_NAMES):
        ax = axes[feat_idx]
        
        # Plot test data
        test_data = test_clean[:, feat_idx]
        ax.hist(test_data, bins=50, alpha=0.5, label='Test', density=True, color='black')
        
        # Plot exemplar data
        colors = plt.cm.tab10(np.linspace(0, 1, len(exemplar_data)))
        for (category, data), color in zip(exemplar_data.items(), colors):
            exemplar_clean = remove_padding(data['features'])
            exemplar_feat = exemplar_clean[:, feat_idx]
            ax.hist(exemplar_feat, bins=50, alpha=0.3, label=category, density=True, color=color)
        
        ax.set_title(f'Distribution of {feat_name}')
        ax.set_xlabel(feat_name)
        ax.set_xscale('log' if feat_name in ['d_in1', 'd_in2', 'd_in3', 'd_out1', 'd_out2', 'd_out3'] else 'linear')
        ax.set_ylabel('Density')
        ax.legend(loc='best', fontsize=8)
        
    # Remove empty subplots
    for idx in range(feat_idx + 1, len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'feature_distributions_all.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved feature distributions to {output_dir}/feature_distributions_all.png")

def plot_feature_boxplots(test_features, exemplar_data, output_dir=analysis_plot_dir):
    """Create box plots for feature comparisons"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Prepare data for plotting
    test_clean = remove_padding(test_features)
    
    # Create individual plots for each feature
    for feat_idx, feat_name in enumerate(FEATURE_NAMES):
        plt.figure(figsize=(12, 6))
        
        # Collect data for box plot
        all_data = []
        labels = []
        
        # Add test data
        all_data.append(test_clean[:, feat_idx])
        labels.append('Test')
        
        # Add exemplar data
        for category, data in exemplar_data.items():
            exemplar_clean = remove_padding(data['features'])
            all_data.append(exemplar_clean[:, feat_idx])
            labels.append(category)
        
        # Create box plot
        plt.boxplot(all_data, labels=labels, showfliers=False)
        plt.title(f'Box Plot: {feat_name}')
        plt.ylabel(feat_name)
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'boxplot_{feat_name}.png'), dpi=150, bbox_inches='tight')
        plt.close()
    
    print(f"Saved box plots to {output_dir}/boxplot_*.png")

def plot_label_distributions(test_labels, exemplar_data, output_dir=analysis_plot_dir):
    """Plot distributions of labels (resource metrics)"""
    os.makedirs(output_dir, exist_ok=True)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for label_idx, label_name in enumerate(LABEL_NAMES):
        ax = axes[label_idx]
        
        # Plot test data
        test_data = test_labels[:, label_idx]
        # Use log scale for better visualization of resource metrics
        if label_name != 'strategy':
            test_data_log = np.log10(test_data + 1)  # Add 1 to avoid log(0)
            ax.hist(test_data_log, bins=50, alpha=0.5, label='Test', density=True, color='black')
            
            # Plot exemplar data
            colors = plt.cm.tab10(np.linspace(0, 1, len(exemplar_data)))
            for (category, data), color in zip(exemplar_data.items(), colors):
                exemplar_label = data['labels'][:, label_idx]
                exemplar_log = np.log10(exemplar_label + 1)
                ax.hist(exemplar_log, bins=50, alpha=0.3, label=category, density=True, color=color)
            
            ax.set_xlabel(f'log10({label_name} + 1)')
        else:
            ax.hist(test_data, bins=50, alpha=0.5, label='Test', density=True, color='black')
            for (category, data), color in zip(exemplar_data.items(), colors):
                exemplar_label = data['labels'][:, label_idx]
                ax.hist(exemplar_label, bins=50, alpha=0.3, label=category, density=True, color=color)
            ax.set_xlabel(label_name)
        
        ax.set_title(f'Distribution of {label_name}')
        ax.set_ylabel('Density')
        ax.legend(loc='best', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'label_distributions.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved label distributions to {output_dir}/label_distributions.png")

def compute_statistical_tests(test_features, exemplar_data):
    """Perform statistical tests comparing test to each exemplar dataset"""
    test_clean = remove_padding(test_features)
    
    results = []
    
    for category, data in exemplar_data.items():
        exemplar_clean = remove_padding(data['features'])
        
        for feat_idx, feat_name in enumerate(FEATURE_NAMES):
            test_data = test_clean[:, feat_idx]
            exemplar_feat = exemplar_clean[:, feat_idx]
            
            # Kolmogorov-Smirnov test
            ks_stat, ks_pvalue = stats.ks_2samp(test_data, exemplar_feat)
            
            # Mann-Whitney U test
            mw_stat, mw_pvalue = stats.mannwhitneyu(test_data, exemplar_feat, alternative='two-sided')
            
            results.append({
                'Category': category,
                'Feature': feat_name,
                'KS_statistic': ks_stat,
                'KS_pvalue': ks_pvalue,
                'MW_statistic': mw_stat,
                'MW_pvalue': mw_pvalue,
                'Significant_KS': ks_pvalue < 0.05,
                'Significant_MW': mw_pvalue < 0.05
            })
    
    return pd.DataFrame(results)

def create_summary_statistics_table(test_features, test_labels, exemplar_data, output_dir=analysis_plot_dir):
    """Create summary statistics tables"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Feature statistics
    test_clean = remove_padding(test_features)
    stats_list = [compute_feature_statistics(test_clean, "Test")]
    
    for category, data in exemplar_data.items():
        exemplar_clean = remove_padding(data['features'])
        stats_list.append(compute_feature_statistics(exemplar_clean, category))
    
    # Create DataFrame and save
    stats_df = pd.DataFrame(stats_list)
    stats_df.to_csv(os.path.join(output_dir, 'feature_statistics_summary.csv'), index=False)
    
    # Label statistics
    label_stats = []
    label_stats.append({
        'Dataset': 'Test',
        'N_models': len(test_labels),
        **{f'{label}_mean': np.mean(test_labels[:, i]) for i, label in enumerate(LABEL_NAMES)},
        **{f'{label}_std': np.std(test_labels[:, i]) for i, label in enumerate(LABEL_NAMES)},
        **{f'{label}_median': np.median(test_labels[:, i]) for i, label in enumerate(LABEL_NAMES)}
    })
    
    for category, data in exemplar_data.items():
        labels = data['labels']
        label_stats.append({
            'Dataset': category,
            'N_models': len(labels),
            **{f'{label}_mean': np.mean(labels[:, i]) for i, label in enumerate(LABEL_NAMES)},
            **{f'{label}_std': np.std(labels[:, i]) for i, label in enumerate(LABEL_NAMES)},
            **{f'{label}_median': np.median(labels[:, i]) for i, label in enumerate(LABEL_NAMES)}
        })
    
    label_stats_df = pd.DataFrame(label_stats)
    label_stats_df.to_csv(os.path.join(output_dir, 'label_statistics_summary.csv'), index=False)
    
    print(f"Saved statistics summaries to {output_dir}/*.csv")
    
    return stats_df, label_stats_df

def plot_correlation_heatmaps(test_features, exemplar_data, output_dir=analysis_plot_dir):
    """Create correlation heatmaps for features"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Test data correlation
    test_clean = remove_padding(test_features)
    test_df = pd.DataFrame(test_clean, columns=FEATURE_NAMES)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(test_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title('Feature Correlation Heatmap - Test Dataset')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'correlation_heatmap_test.png'), dpi=150, bbox_inches='tight')
    plt.close()
    
    # Exemplar correlations
    for category, data in exemplar_data.items():
        exemplar_clean = remove_padding(data['features'])
        exemplar_df = pd.DataFrame(exemplar_clean, columns=FEATURE_NAMES)
        
        plt.figure(figsize=(12, 10))
        sns.heatmap(exemplar_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
                    square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
        plt.title(f'Feature Correlation Heatmap - {category} Dataset')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'correlation_heatmap_{category}.png'), dpi=150, bbox_inches='tight')
        plt.close()
    
    print(f"Saved correlation heatmaps to {output_dir}/correlation_heatmap_*.png")

def main():
    
    if not os.path.exists(analysis_plot_dir):
        os.makedirs(analysis_plot_dir)
    """Main analysis function"""
    print("Loading data...")
    test_features, test_labels, exemplar_data = load_data()
    
    print("\nCreating distribution plots...")
    plot_feature_distributions(test_features, exemplar_data)
    
    print("\nCreating box plots...")
    plot_feature_boxplots(test_features, exemplar_data)
    
    print("\nCreating label distribution plots...")
    plot_label_distributions(test_labels, exemplar_data)
    
    print("\nComputing statistical tests...")
    test_results = compute_statistical_tests(test_features, exemplar_data)
    test_results.to_csv(f'{analysis_plot_dir}/statistical_tests_results.csv', index=False)
    print(f"Statistical test results saved to {analysis_plot_dir}/statistical_tests_results.csv")
    
    # Show summary of significant differences
    significant_features = test_results[test_results['Significant_KS']].groupby('Feature')['Category'].apply(list).to_dict()
    print("\nFeatures with significant differences (KS test, p < 0.05):")
    for feat, categories in significant_features.items():
        print(f"  {feat}: {', '.join(categories)}")
    
    print("\nCreating summary statistics...")
    feature_stats, label_stats = create_summary_statistics_table(test_features, test_labels, exemplar_data, output_dir=analysis_plot_dir)
    
    print("\nCreating correlation heatmaps...")
    plot_correlation_heatmaps(test_features, exemplar_data, output_dir=analysis_plot_dir)

    print(f"\nAnalysis complete! Check the '{analysis_plot_dir}' directory for all outputs.")

if __name__ == "__main__":
    main()

Loading data...
Loaded Jet: features shape (124, 8, 18), labels shape (124, 6)
Loaded Quark: features shape (126, 4, 18), labels shape (126, 6)
Loaded Anomaly: features shape (133, 10, 18), labels shape (133, 6)
Loaded BiPC: features shape (119, 12, 18), labels shape (119, 6)
Loaded Cookie: features shape (130, 8, 18), labels shape (130, 6)
Loaded Automlp: features shape (127, 8, 18), labels shape (127, 6)
Loaded Particle: features shape (128, 8, 18), labels shape (128, 6)

Test data: features shape (94430, 47, 18), labels shape (94430, 6)

Creating distribution plots...
Saved feature distributions to Plotting_distribution_Oct1/feature_distributions_all.png

Creating box plots...
Saved box plots to Plotting_distribution_Oct1/boxplot_*.png

Creating label distribution plots...
Saved label distributions to Plotting_distribution_Oct1/label_distributions.png

Computing statistical tests...
Statistical test results saved to Plotting_distribution_Oct1/statistical_tests_results.csv

Features 